# PiL-HQUC — Colab GPU API only

Notebook này chỉ chạy **FastAPI + Qamomile/CUDA-Q trên GPU Colab** và tạo một URL HTTPS công khai để frontend Vite chạy trên máy local gọi vào.

Luồng chạy:

```text
Frontend local (:5173)
        ↓ HTTPS
Cloudflare Quick Tunnel
        ↓
FastAPI Colab (:8000)
        ↓
Qamomile → CUDA-Q → NVIDIA GPU
```


In [ ]:
# 1) Upload project ZIP
from google.colab import files
from pathlib import Path
import os, shutil, zipfile

os.chdir('/content')
uploaded = files.upload()
zip_names = [name for name in uploaded if name.lower().endswith('.zip')]
if not zip_names:
    raise RuntimeError('Please upload the PiL-HQUC project ZIP.')

zip_path = Path('/content') / zip_names[0]
extract_dir = Path('/content/pil_hquc_api')
if extract_dir.exists():
    shutil.rmtree(extract_dir)
extract_dir.mkdir(parents=True)

with zipfile.ZipFile(zip_path) as archive:
    archive.extractall(extract_dir)

candidates = [p for p in extract_dir.iterdir() if p.is_dir() and (p / 'backend').exists()]
if len(candidates) != 1:
    raise RuntimeError(f'Could not identify project root. Candidates: {candidates}')

ROOT = candidates[0]
BACKEND = ROOT / 'backend'
print('Project root:', ROOT)
print('Backend:', BACKEND)


In [ ]:
# 2) Install CUDA-Q/Qamomile backend dependencies
import subprocess, sys

requirements = BACKEND / 'requirements-quantum-colab.txt'
if not requirements.exists():
    raise FileNotFoundError(requirements)

subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-r', str(requirements)],
    cwd='/content',
    check=True,
)


In [ ]:
# 3) Require NVIDIA GPU target
import os
os.environ['CUDAQ_TARGET'] = 'nvidia'
os.environ['REQUIRE_CUDAQ'] = '1'

import cudaq
cudaq.set_target('nvidia')
target = cudaq.get_target()
name_value = getattr(target, 'name', str(target))
target_name = name_value() if callable(name_value) else str(name_value)
print('CUDA-Q target:', target_name)
assert 'nvidia' in target_name.lower(), target_name


In [ ]:
# 4) Start FastAPI on Colab port 8000
import subprocess, sys, time, requests
from pathlib import Path

# Stop processes created by an earlier run of this notebook.
for proc_name in ('api_process', 'tunnel_process'):
    proc = globals().get(proc_name)
    if proc is not None and proc.poll() is None:
        proc.terminate()
        try:
            proc.wait(timeout=5)
        except subprocess.TimeoutExpired:
            proc.kill()

api_log_path = Path('/content/pil_hquc_api.log')
api_log = api_log_path.open('w')

env = os.environ.copy()
env['PYTHONPATH'] = str(BACKEND)
env['CUDAQ_TARGET'] = 'nvidia'
env['REQUIRE_CUDAQ'] = '1'

api_process = subprocess.Popen(
    [
        sys.executable, '-m', 'uvicorn', 'app.main:app',
        '--host', '0.0.0.0', '--port', '8000',
    ],
    cwd=str(BACKEND),
    env=env,
    stdout=api_log,
    stderr=subprocess.STDOUT,
)

for _ in range(60):
    if api_process.poll() is not None:
        raise RuntimeError(api_log_path.read_text(errors='ignore'))
    try:
        response = requests.get('http://127.0.0.1:8000/api/health', timeout=2)
        if response.ok:
            print('Local API health:', response.json())
            break
    except requests.RequestException:
        pass
    time.sleep(1)
else:
    raise TimeoutError(api_log_path.read_text(errors='ignore'))


In [ ]:
# 5) Create a public HTTPS tunnel for the API (no API key required)
import platform, re, stat, urllib.request

machine = platform.machine().lower()
if machine not in {'x86_64', 'amd64'}:
    raise RuntimeError(f'Unsupported Colab architecture: {machine}')

cloudflared = Path('/content/cloudflared')
if not cloudflared.exists():
    urllib.request.urlretrieve(
        'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64',
        cloudflared,
    )
    cloudflared.chmod(cloudflared.stat().st_mode | stat.S_IEXEC)

cf_log_path = Path('/content/cloudflared.log')
cf_log = cf_log_path.open('w')
tunnel_process = subprocess.Popen(
    [str(cloudflared), 'tunnel', '--url', 'http://127.0.0.1:8000', '--no-autoupdate'],
    cwd='/content',
    stdout=cf_log,
    stderr=subprocess.STDOUT,
)

public_url = None
pattern = re.compile(r'https://[a-z0-9-]+\.trycloudflare\.com')
for _ in range(60):
    if tunnel_process.poll() is not None:
        raise RuntimeError(cf_log_path.read_text(errors='ignore'))
    text = cf_log_path.read_text(errors='ignore')
    match = pattern.search(text)
    if match:
        public_url = match.group(0)
        break
    time.sleep(1)

if not public_url:
    raise TimeoutError(cf_log_path.read_text(errors='ignore'))

API_BASE_URL = public_url + '/api'
print('\nPUBLIC API BASE URL:')
print(API_BASE_URL)
print('\nPut this exact line in frontend/.env.local:')
print(f'VITE_API_BASE_URL={API_BASE_URL}')


In [ ]:
# 6) Verify the public API
health_url = API_BASE_URL + '/health'
response = requests.get(health_url, timeout=30)
print('Status:', response.status_code)
print(response.json())
response.raise_for_status()


## Chạy frontend trên máy local

Trong thư mục `frontend`, tạo file `.env.local`:

```env
VITE_API_BASE_URL=https://<URL-được-in-ở-cell-trên>.trycloudflare.com/api
```

Sau đó **dừng và chạy lại Vite**:

```bash
cd frontend
npm install
npm run dev
```

Mở `http://localhost:5173`.

Lưu ý:

- Không thêm dấu `/` ở cuối URL.
- URL tunnel thay đổi sau mỗi lần Colab runtime/tunnel khởi động lại.
- Giữ notebook và Colab runtime đang hoạt động trong lúc demo.
- Không cần chạy Vite trong Colab.
